In [ ]:
# import the required libraries
import pandas as pd
import torch
import json

In [ ]:
# read the input data (cleaned embedded keywords-title-overview dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title-overview dataset is the dataset 
formed by merging & cleaning the embedded keywords, embedded title and embedded overview dataset.
"""
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/pre-final_mkg_dataset_bert.csv")

<ipython-input-5-2dd0f3d4e44b>:1: DtypeWarning: Columns (0,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/pre-final_mkg_dataset_bert.csv")


In [ ]:
movies_keywords_genres.head(1)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...,[ 3.44745591e-02 -2.66509261e-02 -2.78230965e-...


In [ ]:
movies_keywords_genres.shape[0]

1522874

In [ ]:
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...,[ 3.44745591e-02 -2.66509261e-02 -2.78230965e-...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.3,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...,[-3.16557437e-01 -1.12028293e-01 2.61700094e-...
2,False,1116.0,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.4,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...,[-1.01136873e-02 1.58037364e-01 1.00762844e-...
3,False,1123.0,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...,[-1.07681625e-01 -8.75450373e-02 -9.33825895e-...
4,False,1124.0,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...,[ 1.85870528e-01 1.91051021e-01 5.43662846e-...


In [ ]:
movies_keywords_genres['overview_embedding_string'] = movies_keywords_genres['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

In [ ]:
movies_keywords_genres.loc[0, 'overview_embedding_string']

' 3.44745591e-02 -2.66509261e-02 -2.78230965e-01 -5.36526702e-02  3.31405818e-01  3.90168071e-01  1.46535873e-01  3.00040394e-01  7.53507614e-02  8.83810520e-02 -1.86862782e-01 -2.25032076e-01  1.62307709e-01  5.38947821e-01 -5.10466039e-01  2.89864540e-01  3.88390243e-01  5.53409718e-02 -1.41214535e-01  2.47930184e-01 -2.23860309e-01 -2.47946993e-01 -1.05556794e-01  5.74902117e-01  5.37896991e-01  1.67223692e-01  1.05287954e-01  2.45675147e-01 -2.02234790e-01 -1.59001812e-01  1.57397583e-01  9.93357226e-02 -2.07460612e-01 -2.20275894e-01 -2.00817451e-01  2.30230898e-01  7.95145333e-02 -2.33427539e-01  6.60995468e-02  3.24228793e-01 -7.01512277e-01 -4.16964978e-01 -1.97174013e-01  1.47538468e-01 -2.31593356e-01 -2.07164615e-01  2.48430386e-01  2.36640051e-01  3.05083364e-01 -1.42193124e-01 -4.46930259e-01  4.24459316e-02  4.61624451e-02 -2.37143964e-01  9.77546871e-02  4.09677297e-01 -9.54824314e-02 -2.59163499e-01 -2.91868180e-01 -7.00283423e-02  7.70091861e-02  1.91241801e-01  1.1178

In [ ]:
movies_keywords_genres.head(2)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding,overview_embedding_string
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...,[ 3.44745591e-02 -2.66509261e-02 -2.78230965e-...,3.44745591e-02 -2.66509261e-02 -2.78230965e-0...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.3,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...,[-3.16557437e-01 -1.12028293e-01 2.61700094e-...,-3.16557437e-01 -1.12028293e-01 2.61700094e-0...


In [ ]:
drop_columns = ['overview_embedding']
movies_keywords_genres = movies_keywords_genres.drop(columns=drop_columns)
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding_string
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...,3.44745591e-02 -2.66509261e-02 -2.78230965e-0...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.3,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...,-3.16557437e-01 -1.12028293e-01 2.61700094e-0...
2,False,1116.0,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.4,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...,-1.01136873e-02 1.58037364e-01 1.00762844e-0...
3,False,1123.0,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...
4,False,1124.0,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...,1.85870528e-01 1.91051021e-01 5.43662846e-0...


In [ ]:
nan_count = movies_keywords_genres['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
movies_keywords_genres.dropna(subset=['overview_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding_string' has 1288162 NaN values.
The column 'genres' has 1179649 NaN values.


In [ ]:
movies_keywords_genres.shape[0]

234712

In [ ]:
# this is the final embedded dataset
movies_keywords_genres.to_csv('/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/final_mkg_dataset_bert.csv', index=False)